# Convolutional Neural Network

### Importing the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

2025-11-05 11:17:06.021772: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-05 11:17:06.073080: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-05 11:17:07.914251: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
tf.__version__

'2.20.0'

## Part 1 - Data Preprocessing

why  transformation? 
- to prevent overfitting 
- high train accuracy and low test accuracy if not 


In [3]:
# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    rotation_range=20,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True
)

training_set = train_datagen.flow_from_directory(
    '001dataset/dataset/training_set',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 8000 images belonging to 2 classes.


### Preprocessing TestSet

In [4]:
# Only rescaling for test set
test_datagen = ImageDataGenerator(rescale=1./255)

test_set = test_datagen.flow_from_directory(
    '001dataset/dataset/test_set',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.


## Part 2 - Building the CNN

Initializing the CNN

In [5]:
cnn = tf.keras.models.Sequential()

Step 1 - Convolution and pooling

In [6]:
# 1st Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

/home/kygiet/anaconda3/envs/MLapps/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1762320729.438883   63734 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1762320729.454863   63734 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
# 2nd Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(64, (3,3), activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

In [8]:
# 3rd Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(128, (3,3), activation='relu'))
cnn.add(tf.keras.layers.MaxPooling2D(pool_size=(2,2)))

### Step 2 - Flattening 

In [9]:
cnn.add(tf.keras.layers.Flatten())

### step 4 - Full Connection 

In [10]:
# Full connection + Dropout
cnn.add(tf.keras.layers.Dense(128, activation='relu'))
cnn.add(tf.keras.layers.Dropout(0.5))

### step 5 - Output layer

In [11]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [12]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [13]:
# EarlyStopping to prevent overfitting
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

### Training the CNN on the Training Set and evaluationg it on the Test Set 

In [14]:
cnn.fit(
    x=training_set,
    validation_data=test_set,
    epochs=50,  # Train longer
    callbacks=[es]
)

/home/kygiet/anaconda3/envs/MLapps/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50


2025-11-05 11:17:11.328057: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.


  1/250 ━━━━━━━━━━━━━━━━━━━━ 7:08 2s/step - accuracy: 0.3438 - loss: 0.6997

2025-11-05 11:17:11.562365: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.
2025-11-05 11:17:11.634410: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.


  2/250 ━━━━━━━━━━━━━━━━━━━━ 1:00 245ms/step - accuracy: 0.3906 - loss: 0.9054

2025-11-05 11:17:11.807987: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.
2025-11-05 11:17:11.890137: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 65028096 exceeds 10% of free system memory.


250/250 ━━━━━━━━━━━━━━━━━━━━ 121s 481ms/step - accuracy: 0.5109 - loss: 0.7123 - val_accuracy: 0.5765 - val_loss: 0.6905
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 117s 467ms/step - accuracy: 0.6062 - loss: 0.6680 - val_accuracy: 0.6690 - val_loss: 0.6296
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 110s 441ms/step - accuracy: 0.6606 - loss: 0.6221 - val_accuracy: 0.7225 - val_loss: 0.5622
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 103s 413ms/step - accuracy: 0.7035 - loss: 0.5754 - val_accuracy: 0.7520 - val_loss: 0.5096
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 103s 413ms/step - accuracy: 0.7356 - loss: 0.5278 - val_accuracy: 0.7690 - val_loss: 0.4952
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 110s 439ms/step - accuracy: 0.7593 - loss: 0.4976 - val_accuracy: 0.7830 - val_loss: 0.4535
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 109s 438ms/step - accuracy: 0.7691 - loss: 0.4742 - val_accuracy: 0.7940 - val_loss: 0.4392
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 110s 433ms/step - accuracy: 0.7847 - loss: 0.46

## Making a single Prediction

In [17]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    test_image = image.load_img(img_path, target_size=(128,128))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)/255.0
    result = cnn.predict(test_image)
    prediction = 'dog' if result[0][0] > 0.5 else 'cat'
    print(f"Prediction: {prediction}, Confidence: {result[0][0]:.3f}")

# Example usage
predict_image('./cat.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Prediction: cat, Confidence: 0.379


In [16]:
print(training_set.class_indices)
print(training_set.samples)


{'cats': 0, 'dogs': 1}
8000
